In [2]:
from dotenv import load_dotenv
from pathlib import Path
import os

# remonter à la racine du projet
env_path = Path("..") / ".env"

load_dotenv(env_path)

print(os.getenv("MISTRAL_API_KEY"))



osZDGy23YzF4yFZAlKw6OCF03NZcnDKt


# 1 Connexion à Qdrant

In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='rag_pedagogique')])

In [4]:
collection_info = client.get_collection('rag_pedagogique')
print("Collection info:", collection_info)

Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=126 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), 

In [5]:
# =========================================================
# 0. IMPORTS
# =========================================================
from qdrant_client import QdrantClient
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import Qdrant
from langchain_core.documents import Document

collection_name = "rag_pedagogique"

# =========================================================
# 2. MISTRAL EMBEDDINGS (IMPORTANT)
# =========================================================
embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

# =========================================================
# 3. RERANKER
# =========================================================
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# =========================================================
# 4. LLM
# =========================================================
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [10]:
from qdrant_client import models

# =========================================================
# 5. PROMPT
# =========================================================
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant chargé d'extraire des informations.

RÈGLES :
- utilise uniquement le contexte
- copie exactement le texte
- si rien : "je ne sais pas"
- ne reformule jamais 
- ne modifie pas les titres ses sections
                                          

Contexte :
{context}

Question :
{question}

Réponse :
""")

chain = prompt | llm

# =========================================================
# 6. QUESTION ref
# =========================================================
question = "tu peux me dire quels sont les capacités attendues du cours sur la dérivée  "

rewrite_prompt = ChatPromptTemplate.from_template("""
Tu es un expert du Bulletin Officiel (BO) de mathématiques du lycée français.

Ta tâche est de transformer la question d’un utilisateur en une requête optimisée pour une recherche dans un corpus du BO.

Le BO est structuré en sections officielles :
- Contenus
- Capacités attendues
- Démonstrations
- Exemples d’algorithmes
- Approfondissements possibles

Règles :
- Remplace les mots de l’utilisateur par les intitulés officiels du BO
  (ex : "compétences" → "capacités attendues")
- Identifie le chapitre concerné (ex : trigonométrie)
- Ne reformule PAS librement : normalise vers le vocabulaire BO
- Ne rajoute aucune information
- Produit UNE seule requête optimisée pour recherche vectorielle                                             

Question utilisateur :
{question}

Requête BO optimisée :
""")

chain_rewrite_prompt = rewrite_prompt | llm

rewritten_question = chain_rewrite_prompt.invoke({
    "question": question,
})

print(rewritten_question.content)
# =========================================================
# 7. EMBEDDING QUERY (MÊME MODÈLE QUE INGESTION)
# =========================================================
query_vector = embeddings.embed_query(rewritten_question.content)

# =========================================================
# 8. SEARCH QDRANT
# =========================================================

client.create_payload_index(
    collection_name=collection_name,
    field_name="metadata.classe",
    field_schema="keyword"
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="metadata.voie",
    field_schema="keyword"
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="metadata.annee",
    field_schema="keyword"
)

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=30, # Return the top 1 most similar vector
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.classe",
                match=models.MatchValue(value="premiere")
            ),
            models.FieldCondition(
                key="metadata.voie",
                match=models.MatchValue(value="generale")
            ),
            models.FieldCondition(
                key="metadata.annee",
                match=models.MatchValue(value="2026")
            ),
        ]
    ),
)

docs = [
    Document(
        page_content=hit.payload["page_content"],
        metadata=hit.payload["metadata"]
    )
    for hit in search_results.points
]

# =========================================================
# 11. DEBUG
# =========================================================

print("\n===== DOCS FILTRÉS =====\n")

for d in docs:
    print(d.metadata.get("title"))
    print(d.metadata.get("classe"), d.metadata.get("voie"))
    print(d.page_content[:200])
    print("------")

# =========================================================
# 12. RERANKING
# =========================================================

def rerank_documents(query, docs, top_k=5):
    if not docs:
        return []

    pairs = [(query, doc.page_content[:2000]) for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in ranked[:top_k]]

reranked_docs = rerank_documents(question, docs, top_k=5)

# =========================================================
# 13. CONTEXT FINAL
# =========================================================

context = "\n\n".join(
    doc.page_content for doc in reranked_docs
)

# =========================================================
# 14. LLM CALL
# =========================================================

response = chain.invoke({
    "context": context,
    "question": question,
})

print("\n===== RÉPONSE =====\n")
print(response.content)

capacités attendues dérivée analyse

===== DOCS FILTRÉS =====

BO_lycee_premiere_generale_specialite_2026
premiere generale
tangente à la courbe représentative de ƒ au point d’abscisse a est la droite d’équation 𝑦 = ƒ (a) + ƒ ‘(a)(𝑥 – a).  
− Approximation linéaire : fonction affine tangente 𝑥 ↦ ƒ (a) + ƒ ‘(a)(𝑥 – a) et ap
------
BO_lycee_premiere_generale_scientifique_2026
premiere generale
représentation graphique (diagrammes en barres, diagrammes 
circulaires). 
Analyse statistique de deux caractères quantitatifs. 
Représentation par un nuage de points. 
Ajustement affine, point moyen.
------
BO_lycee_premiere_generale_specialite_2026
premiere generale
Analyse  
Objectifs  
Deux points fondamentaux du programme de première sont ici étudiés : le concept de dérivée, avec ses applications à l’étude 
des fonctions, et la fonction exponentielle.  
L’étud
------
BO_lycee_premiere_generale_scientifique_2026
premiere generale
Évolutions et variations 
− Appliquer un taux d’évolution pour ca